In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve2d

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [3]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/2024-03-19/fdt/*'))
files_fdt

['/home/ulyanov/data/cross/2024-03-19/fdt/solo_L2_phi-fdt-blos_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/2024-03-19/fdt/solo_L2_phi-fdt-icnt_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/2024-03-19/fdt/solo_L2_phi-fdt-stokes_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/2024-03-19/fdt/solo_L2_phi-fdt-vlos_20240319T152009_V202602220014_0443190639.fits.gz']

In [4]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/2024-03-19/hmi/*'))
files_hmi

['/home/ulyanov/data/cross/2024-03-19/hmi/hmi.Ic_45s.20240319_152445_TAI.2.continuum.fits',
 '/home/ulyanov/data/cross/2024-03-19/hmi/hmi.M_45s.20240319_152445_TAI.2.magnetogram.fits']

In [5]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xu, yu = s['xu'], s['yu']

In [6]:
file_hmi = files_hmi[0]
file_fdt = files_fdt[1]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

binning = header_hmi['RSUN_OBS'] / header_hmi['CDELT1'] / (header_fdt['RSUN_ARC'] / header_fdt['CDELT1'])
print(binning)

data_hmi = rebin(data_hmi, int(round(binning)), update_header=header_hmi)

data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1,
                     xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                     crota=header_fdt['CROTA'] + 0.25,
                     grid=crop_grid(xu, yu, header_fdt),
                     kind='quadratic')

data_fdt[np.isnan(data_hmi)] = np.nan

#data_hmi = data_hmi[897-16:897+16, 1155-16:1155+16]
#data_fdt = data_fdt[897-16:897+16, 1155-16:1155+16]


3.140790459562361


In [8]:
ft_fdt = np.fft.fft2(np.nan_to_num(data_fdt))
ft_hmi = np.fft.fft2(np.nan_to_num(data_hmi))

#ft_psf = ft_fdt * np.conj(ft_hmi) / (np.abs(ft_hmi) ** 2 + 1e-4)
ft_psf = ft_hmi * np.conj(ft_fdt) / (np.abs(ft_fdt) ** 2 + 1e-4)
psf = np.real(np.fft.ifft2(ft_psf))
psf = np.roll(psf, (4, 4), axis=(0,1))
psf = psf[:9,:9]#.clip(0)
psf /= np.sum(psf)

plt.figure(figsize=(10,10))
plt.imshow(psf, cmap='gray')

In [43]:
data_hmi_ = convolve2d(data_hmi, psf, mode='same', boundary='symm')

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, origin='lower')
#plt.axis('off')

#plt.ylim(897-50,897+50)
#plt.xlim(1155-50,1155+50)

plt.xlabel('X, pixels')
plt.ylabel('Y, pixels')
plt.title('FDT')

plt.tight_layout()

In [51]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, origin='lower')
#plt.axis('off')

#plt.ylim(897-50,897+50)
#plt.xlim(1155-50,1155+50)

plt.xlabel('X, pixels')
plt.ylabel('Y, pixels')
plt.title('HMI')

plt.tight_layout()

In [52]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi_, origin='lower')
#plt.axis('off')

#plt.ylim(897-50,897+50)
#plt.xlim(1155-50,1155+50)

plt.xlabel('X, pixels')
plt.ylabel('Y, pixels')
plt.title('HMI')

plt.tight_layout()